## 8. Train/Val/Test Split

- 80/10/10 비율, 카테고리 층화 추출 (stratified split)
- Test set: 최소 300건, 카테고리당 최소 30건
- 카테고리별 최소 건수 미달 시 오버샘플링으로 보정

In [ ]:
# ============================================================
# Cell 10: Instruction Format Generation
# ============================================================

SYSTEM_MESSAGE = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."
INSTRUCTION = "다음 민원에 대해 공손하고 명확한 답변을 작성하세요."

# 카테고리별 의미 있는 thought 템플릿
THOUGHT_TEMPLATES = {
    "교통": "이 민원은 교통 분야와 관련됩니다. 도로, 대중교통, 주차, 교통안전 등의 관련 정책과 담당 부서를 확인하고, 민원인에게 구체적인 진행 상황과 계획을 안내해야 합니다.",
    "환경": "이 민원은 환경 분야와 관련됩니다. 환경미화, 상하수도, 폐기물, 소음, 공원녹지 등 관련 조례와 처리 절차를 확인하고, 적절한 조치 방안을 안내해야 합니다.",
    "복지": "이 민원은 복지/보건 분야와 관련됩니다. 사회복지, 보건의료, 노인/아동/장애인 복지 등 관련 제도와 지원 자격 요건을 확인하고, 신청 방법과 절차를 안내해야 합니다.",
    "건축": "이 민원은 건축/도시계획 분야와 관련됩니다. 건축허가, 개발행위, 도시계획, 토지이용 등 관련 법령과 행정 절차를 확인하고, 필요한 서류와 처리 기한을 안내해야 합니다.",
    "행정": "이 민원은 일반 행정 분야와 관련됩니다. 민원 접수, 증명서 발급, 행정 절차 등 관련 규정을 확인하고, 처리 방법과 담당 부서 연락처를 안내해야 합니다.",
    "세금": "이 민원은 세무/재정 분야와 관련됩니다. 지방세, 과세 기준, 감면 요건, 납부 방법 등 관련 세법과 조례를 확인하고, 정확한 세금 정보를 안내해야 합니다.",
    "안전": "이 민원은 안전/재난 분야와 관련됩니다. 재난안전, 소방, 시설물 안전, 민방위 등 관련 법령과 비상 대응 절차를 확인하고, 안전 조치 사항을 안내해야 합니다.",
    "기타": "이 민원의 내용을 분석하여 관련 부서와 정책을 확인하고, 민원인에게 적절한 처리 방안과 담당 부서 정보를 안내해야 합니다.",
}


def format_instruction_record(rec: dict, include_thought: bool = False) -> dict:
    """레코드를 instruction 형식으로 변환."""
    category = rec["category"]
    question = rec["question"]
    answer = rec["answer"]
    
    # Input 구성
    input_text = f"[카테고리: {category}]\n민원 내용: {question}"
    
    # Output 구성
    if include_thought and category in THOUGHT_TEMPLATES:
        thought = THOUGHT_TEMPLATES[category]
        output_text = f"<thought>\n{thought}\n</thought>\n{answer}"
    else:
        output_text = answer
    
    return {
        "id": rec["id"],
        "system": SYSTEM_MESSAGE,
        "instruction": INSTRUCTION,
        "input": input_text,
        "output": output_text,
        "category": category,
        "source": rec.get("source_dataset", ""),
    }


# --- 71852 + 98 데이터를 instruction 형식으로 변환 ---
# Q&A 쌍이 있는 데이터만 사용 (619는 Q-only이므로 별도 저장)
all_qa_records = records_71852 + records_98

print(f"=== Instruction Format 변환 ===")
print(f"Q&A 레코드 수: {len(all_qa_records)} (71852: {len(records_71852)}, 98: {len(records_98)})")

# 중복 제거 (질문 해시 기반)
seen_hashes = set()
unique_records = []

for rec in all_qa_records:
    q_hash = hashlib.md5(rec["question"].encode()).hexdigest()
    if q_hash not in seen_hashes:
        seen_hashes.add(q_hash)
        unique_records.append(rec)

print(f"중복 제거 후: {len(unique_records)} (제거: {len(all_qa_records) - len(unique_records)})")

# Instruction 형식 변환 (thought 없는 버전 - 기본)
formatted_records = []
for rec in tqdm(unique_records, desc="Instruction 변환"):
    formatted = format_instruction_record(rec, include_thought=False)
    formatted_records.append(formatted)

# Instruction 형식 변환 (thought 있는 버전 - 별도 저장)
formatted_records_with_thought = []
for rec in unique_records:
    formatted = format_instruction_record(rec, include_thought=True)
    formatted_records_with_thought.append(formatted)

print(f"\n최종 instruction 레코드: {len(formatted_records)}")

# 카테고리별 통계
df_formatted = pd.DataFrame(formatted_records)
print(f"\n카테고리 분포:")
print(df_formatted["category"].value_counts())

# 샘플 출력
print(f"\n--- 최종 형식 샘플 ---")
sample = formatted_records[0]
print(json.dumps(sample, ensure_ascii=False, indent=2)[:1000])

## 7. Instruction Format 생성

최종 학습 데이터 포맷으로 변환합니다.

- **System**: "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."
- **Instruction**: "다음 민원에 대해 공손하고 명확한 답변을 작성하세요."
- **Input**: `[카테고리: {category}]\n민원 내용: {question}`
- **Output**: 전문 정부 답변 (thought 태그 없음, 직접 답변)
- **Output (with thought)**: 카테고리별 의미 있는 사고 과정 + 답변

In [ ]:
# ============================================================
# Cell 9: PII Masking Improvement
# ============================================================

def improve_pii_masking(text: str) -> str:
    """PII 마스킹을 엔티티 단위로 개선.
    
    기존 [NAME_MASKED] -> 의미 있는 엔티티 태그로 변환.
    ○○○, ▲▲▲ 스타일은 유지 (이미 엔티티 단위).
    619 데이터의 #@주소#, #@이름# 스타일도 표준화.
    """
    if not text:
        return text
    
    result = text
    
    # 1. [NAME_MASKED] 연속 패턴을 엔티티 태그로 변환
    # [NAME_MASKED] 가 연속으로 나오는 경우를 하나의 엔티티로 통합
    # 예: "[NAME_MASKED]님[NAME_MASKED]" -> "[이름]님"
    result = re.sub(r'(\[NAME_MASKED\])+', '[이름]', result)
    
    # 2. <NAME> 태그 변환
    result = re.sub(r'<NAME>', '[이름]', result)
    
    # 3. 전화번호 패턴 변환
    result = re.sub(r'<MOBILE_NUMBER>', '[전화번호]', result)
    result = re.sub(r'<PHONE_NUMBER>', '[전화번호]', result)
    
    # 4. 주소 패턴 변환
    result = re.sub(r'<ADDRESS>', '[주소]', result)
    
    # 5. 날짜/시간 패턴 변환
    result = re.sub(r'<DATE>', '[날짜]', result)
    result = re.sub(r'<TIME>', '[시간]', result)
    
    # 6. 금액 패턴 변환
    result = re.sub(r'<CHARGE>', '[금액]', result)
    
    # 7. 생년월일 변환
    result = re.sub(r'<BIRTH_NUMBER>', '[생년월일]', result)
    
    # 8. 619 데이터의 PII 태그 표준화
    result = re.sub(r'#@주소#', '[주소]', result)
    result = re.sub(r'#@이름#', '[이름]', result)
    result = re.sub(r'#@전화번호#', '[전화번호]', result)
    result = re.sub(r'#@생년월일#', '[생년월일]', result)
    result = re.sub(r'#@카드번호#', '[카드번호]', result)
    result = re.sub(r'#@계좌번호#', '[계좌번호]', result)
    
    # 9. ○ 연속 (3개 이상) -> 그대로 유지 (이미 적절한 마스킹)
    # ▲ 연속도 유지
    
    # 10. 연속된 엔티티 태그 정리 (중복 제거)
    result = re.sub(r'(\[이름\])\s*(\[이름\])+', '[이름]', result)
    result = re.sub(r'(\[전화번호\])\s*(\[전화번호\])+', '[전화번호]', result)
    result = re.sub(r'(\[주소\])\s*(\[주소\])+', '[주소]', result)
    
    return result


def calculate_pii_density(text: str) -> float:
    """텍스트의 PII 토큰 밀도를 계산."""
    if not text:
        return 0.0
    total_len = len(text)
    # PII 토큰 패턴들
    pii_patterns = [
        r'\[이름\]', r'\[전화번호\]', r'\[주소\]', r'\[날짜\]',
        r'\[시간\]', r'\[금액\]', r'\[생년월일\]', r'\[카드번호\]',
        r'\[계좌번호\]', r'\[NAME_MASKED\]',
        r'○{2,}', r'▲{2,}',
        r'<NAME>', r'<ADDRESS>', r'<MOBILE_NUMBER>', r'<PHONE_NUMBER>',
        r'<DATE>', r'<CHARGE>', r'<BIRTH_NUMBER>',
    ]
    pii_len = 0
    for pat in pii_patterns:
        for m in re.finditer(pat, text):
            pii_len += len(m.group())
    return pii_len / total_len if total_len > 0 else 0.0


# --- 71852 데이터에 PII 개선 적용 ---
print("=== PII 마스킹 개선 ===\n")

pii_before = []
pii_after = []

for rec in tqdm(records_71852, desc="71852 PII 개선"):
    # Before
    q_density_before = calculate_pii_density(rec["question"])
    a_density_before = calculate_pii_density(rec["answer"])
    pii_before.append((q_density_before + a_density_before) / 2)
    
    # Apply improvement
    rec["question"] = improve_pii_masking(rec["question"])
    rec["answer"] = improve_pii_masking(rec["answer"])
    
    # After
    q_density_after = calculate_pii_density(rec["question"])
    a_density_after = calculate_pii_density(rec["answer"])
    pii_after.append((q_density_after + a_density_after) / 2)
    
    # Update lengths
    rec["q_len"] = len(rec["question"])
    rec["a_len"] = len(rec["answer"])

# 98 데이터에도 적용
for rec in records_98:
    rec["question"] = improve_pii_masking(rec["question"])
    rec["answer"] = improve_pii_masking(rec["answer"])
    rec["q_len"] = len(rec["question"])
    rec["a_len"] = len(rec["answer"])

avg_pii_before = np.mean(pii_before) * 100
avg_pii_after = np.mean(pii_after) * 100

print(f"PII 밀도 (Before): {avg_pii_before:.2f}%")
print(f"PII 밀도 (After):  {avg_pii_after:.2f}%")
print(f"감소: {avg_pii_before - avg_pii_after:.2f}%p")

# 샘플 비교
print(f"\n--- PII 개선 전후 비교 ---")
sample_idx = 0
for rec in records_71852[:20]:
    if "[이름]" in rec["answer"] or "○○" in rec["answer"]:
        print(f"Answer (처리 후): {rec['answer'][:200]}")
        print()
        sample_idx += 1
        if sample_idx >= 3:
            break

wandb.log({
    "pii_density_before": avg_pii_before,
    "pii_density_after": avg_pii_after,
    "pii_density_reduction": avg_pii_before - avg_pii_after,
})

## 6. PII 마스킹 개선

기존 문제: `[NAME_MASKED]`가 문자 단위로 적용되어 텍스트 가독성이 파괴됨 (밀도 23.35%)

개선 방향:
- **엔티티 단위 마스킹**: `[이름]`, `[전화번호]`, `[주소]`, `[날짜]` 등 의미 있는 토큰으로 대체
- 71852의 `○○○○` 스타일은 이미 엔티티 단위이므로 유지
- 목표: PII 토큰 밀도 < 5%

In [ ]:
# ============================================================
# Cell 8: Dataset 98 (다산콜센터) - 민원 관련 필터링
# ============================================================

# 다산콜센터 카테고리 -> 표준 카테고리 매핑 (민원 관련만)
DASAN_CATEGORY_MAP = {
    "대중교통 안내": "교통",
    "생활하수도 관련 문의": "환경",
    "일반행정 문의": "행정",
    # "코로나19 관련 상담": 제외 (시의성 낮음)
}

records_98 = []
skipped_98 = 0

for fpath in tqdm(glob.glob(os.path.join(DATASET_98_LABEL, "*.json")), desc="98 처리"):
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except:
        continue
    
    if not isinstance(data, list):
        continue
    
    # 대화셋 단위로 Q&A 쌍 재구성
    # 데이터는 대화 턴 단위로 저장됨 - 대화셋일련번호로 그룹화
    dialog_groups = defaultdict(list)
    for rec in data:
        dialog_id = rec.get("대화셋일련번호", "")
        dialog_groups[dialog_id].append(rec)
    
    for dialog_id, turns in dialog_groups.items():
        # 카테고리 확인
        cat = turns[0].get("카테고리", "")
        if cat not in DASAN_CATEGORY_MAP:
            skipped_98 += 1
            continue
        
        std_category = DASAN_CATEGORY_MAP[cat]
        
        # Q&A 쌍 재구성: 고객질문 + 상담사답변
        questions = []
        answers = []
        
        for turn in sorted(turns, key=lambda x: int(x.get("문장번호", 0))):
            speaker = turn.get("화자", "")
            q_text = turn.get("고객질문(요청)", "").strip()
            a_text = turn.get("상담사답변", "").strip()
            
            if q_text:
                questions.append(q_text)
            if a_text:
                answers.append(a_text)
        
        full_question = " ".join(questions).strip()
        full_answer = " ".join(answers).strip()
        
        if not full_question or len(full_question) < 10:
            continue
        if not full_answer or len(full_answer) < 30:
            continue
        
        records_98.append({
            "id": f"98_{dialog_id}",
            "question": full_question,
            "answer": full_answer,
            "title": "",
            "category": std_category,
            "raw_category": cat,
            "source_dataset": "98",
            "q_len": len(full_question),
            "a_len": len(full_answer),
        })

df_98 = pd.DataFrame(records_98) if records_98 else pd.DataFrame()

print(f"\n=== 98 다산콜센터 필터링 결과 ===")
print(f"민원 관련 Q&A 쌍: {len(records_98)}")
print(f"스킵 (비민원): {skipped_98}")
if len(records_98) > 0:
    print(f"\n카테고리 분포:")
    print(df_98["category"].value_counts())
    print(f"\n답변 길이 통계:")
    print(df_98["a_len"].describe())

wandb.log({"98_filtered_records": len(records_98), "98_skipped": skipped_98})

## 5. 98 다산콜센터 필터링 (민원 관련만)

다산콜센터 데이터 중 실제 지자체 민원과 관련된 대중교통, 생활하수도, 일반행정 건만 필터링합니다.
코로나19 상담은 시의성이 낮으므로 제외합니다. Q&A 쌍으로 재구성합니다.

In [ ]:
# ============================================================
# Cell 7: 619 Data Processing (Q-only, supplementary)
# ============================================================

# 619 카테고리 -> 표준 카테고리 매핑
CATEGORY_619_MAP = {
    "건축허가": "건축",
    "경제": "기타",
    "공통": "행정",
    "교통": "교통",
    "농업_축산": "기타",
    "문화_체육_관광": "행정",
    "보건소": "복지",
    "복지": "복지",
    "산림": "환경",
    "상하수도": "환경",
    "세무": "세금",
    "안전건설": "안전",
    "위생": "환경",
    "자동차": "교통",
    "정보통신": "행정",
    "토지": "건축",
    "행정": "행정",
    "환경미화": "환경",
}

MAX_SAMPLES_PER_CAT = 500  # 카테고리당 최대 샘플 수

records_619 = []

if os.path.exists(DATASET_619_LABEL):
    for cat_dir in tqdm(sorted(os.listdir(DATASET_619_LABEL)), desc="619 처리"):
        cat_path = os.path.join(DATASET_619_LABEL, cat_dir)
        if not os.path.isdir(cat_path):
            continue
        
        std_category = CATEGORY_619_MAP.get(cat_dir, "기타")
        cat_docs = []
        
        for fpath in glob.glob(os.path.join(cat_path, "*.json")):
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    data = json.load(f)
                for doc in data.get("documents", []):
                    q = doc.get("Q_refined", "").strip()
                    if not q or len(q) < 10:
                        continue
                    
                    # NER 엔티티 정보 추출 (PII 개선 참조용)
                    labeling = doc.get("labeling", {})
                    entities = labeling.get("entities", [])
                    intent = labeling.get("intent", {})
                    keywords = labeling.get("keyword", [])
                    
                    cat_docs.append({
                        "id": f"619_{cat_dir}_{doc.get('id', '')}",
                        "question": q,
                        "answer": "",  # Q-only
                        "category": std_category,
                        "raw_category": cat_dir,
                        "subcategory": intent.get("subcategory", ""),
                        "predication": intent.get("predication", ""),
                        "entities": entities,
                        "keywords": [k.get("form", "") for k in keywords],
                        "source_dataset": "619",
                        "q_len": len(q),
                        "a_len": 0,
                    })
            except:
                pass
        
        # 카테고리당 샘플링
        if len(cat_docs) > MAX_SAMPLES_PER_CAT:
            random.shuffle(cat_docs)
            cat_docs = cat_docs[:MAX_SAMPLES_PER_CAT]
        
        records_619.extend(cat_docs)
        print(f"  {cat_dir} -> {std_category}: {len(cat_docs)}건 샘플링")

df_619 = pd.DataFrame(records_619)

print(f"\n=== 619 처리 결과 ===")
print(f"총 샘플: {len(records_619)}")
print(f"\n카테고리 분포:")
print(df_619["category"].value_counts())
print(f"\n질문 길이 통계:")
print(df_619["q_len"].describe())

wandb.log({"619_sampled_records": len(records_619)})

## 4. 619 데이터 처리 (Supplementary - Q only)

619 데이터셋은 Q-only이므로 답변 생성 학습에는 직접 사용하지 않습니다.
분류 학습 보조 및 카테고리별 입력 예시 풍부화를 위해 카테고리당 최대 500건씩 샘플링합니다.

In [ ]:
# ============================================================
# Cell 6: 71852 - 파싱 품질 검증 및 샘플 확인
# ============================================================

print("=== 파싱 결과 샘플 확인 ===\n")

# 카테고리별 샘플 1개씩 출력
for cat in STANDARD_CATEGORIES:
    subset = df_71852[df_71852["category"] == cat]
    if len(subset) == 0:
        print(f"[{cat}] - 데이터 없음\n")
        continue
    sample = subset.iloc[0]
    print(f"[{cat}] (원본: {sample['raw_category']}) - ID: {sample['id']}")
    print(f"  Q ({sample['q_len']}자): {sample['question'][:150]}...")
    print(f"  A ({sample['a_len']}자): {sample['answer'][:200]}...")
    print()

# 답변 길이 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_71852["a_len"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(df_71852["a_len"].median(), color="red", linestyle="--", label=f"Median: {df_71852['a_len'].median():.0f}")
axes[0].axvline(97, color="orange", linestyle="--", label="기존 평균: 97자")
axes[0].set_title("71852 답변 길이 분포 (재구성 후)")
axes[0].set_xlabel("답변 길이 (자)")
axes[0].set_ylabel("빈도")
axes[0].legend()

cat_counts = df_71852["category"].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color="steelblue", alpha=0.8)
axes[1].set_title("71852 카테고리 분포")
axes[1].set_xlabel("건수")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "71852_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

# 매핑되지 않은 원본 카테고리 확인
unmapped = df_71852[df_71852["category"] == "기타"]["raw_category"].value_counts()
print(f"\n'기타'로 매핑된 원본 카테고리 ({len(unmapped)}종):")
print(unmapped.head(20))

In [ ]:
# ============================================================
# Cell 5: 71852 Data Processing - Q/A Parsing & Category Mapping
# ============================================================

# --- 카테고리 매핑 ---
# 71852 원본 카테고리를 표준 8개 카테고리로 매핑
CATEGORY_MAP = {
    # 교통
    "교통": "교통", "교통행정": "교통", "교통과": "교통",
    "대중교통": "교통", "도로교통": "교통", "교통정책": "교통",
    "교통정책과": "교통", "도로과": "교통",
    # 환경
    "환경": "환경", "환경과": "환경", "환경미화": "환경",
    "환경위생": "환경", "환경정책": "환경", "상하수도": "환경",
    "수도": "환경", "하수도": "환경", "청소행정": "환경",
    "공원녹지": "환경", "산림": "환경", "녹지": "환경",
    # 복지
    "복지": "복지", "복지과": "복지", "복지정책": "복지",
    "사회복지": "복지", "보건": "복지", "보건소": "복지",
    "보건의료": "복지", "노인복지": "복지", "아동복지": "복지",
    "장애인복지": "복지", "여성가족": "복지", "주민생활지원": "복지",
    # 건축
    "건축": "건축", "건축과": "건축", "건축허가": "건축",
    "건설": "건축", "도시계획": "건축", "주택": "건축",
    "도시개발": "건축", "건축행정": "건축", "개발행위": "건축",
    "토지": "건축", "부동산": "건축",
    # 행정
    "행정": "행정", "행정과": "행정", "일반행정": "행정",
    "총무": "행정", "민원봉사": "행정", "자치행정": "행정",
    "인사": "행정", "기획": "행정", "감사": "행정",
    "법무": "행정", "홍보": "행정", "문화체육": "행정",
    "문화": "행정", "체육": "행정", "관광": "행정",
    "정보통신": "행정", "전산": "행정",
    # 세금
    "세무": "세금", "세금": "세금", "세무과": "세금",
    "재정": "세금", "회계": "세금", "징수": "세금",
    # 안전
    "안전": "안전", "재난안전": "안전", "안전건설": "안전",
    "소방": "안전", "방재": "안전", "민방위": "안전",
    "안전관리": "안전", "재난": "안전",
    # 기타
    "기타": "기타", "경제": "기타", "농업": "기타",
    "축산": "기타", "수산": "기타", "위생": "기타",
    "자동차": "기타",
}

STANDARD_CATEGORIES = ["교통", "환경", "복지", "건축", "행정", "세금", "안전", "기타"]


def map_category(raw_category: str) -> str:
    """원본 카테고리를 표준 카테고리로 매핑."""
    if not raw_category:
        return "기타"
    raw = raw_category.strip()
    # 정확한 매칭
    if raw in CATEGORY_MAP:
        return CATEGORY_MAP[raw]
    # 부분 매칭
    for key, val in CATEGORY_MAP.items():
        if key in raw or raw in key:
            return val
    return "기타"


def parse_consulting_content(content: str) -> Tuple[str, str, str]:
    """consulting_content에서 제목, Q, A를 파싱.
    
    Returns:
        (title, question, answer) 튜플
    """
    if not content:
        return "", "", ""
    
    title = ""
    question = ""
    answer = ""
    
    # 제목 추출
    title_match = re.search(r"제목\s*:\s*(.+?)(?:\n|Q\s*:)", content, re.DOTALL)
    if title_match:
        title = title_match.group(1).strip()
    
    # Q와 A 분리 - "Q :" 또는 "Q:" 마커 사용
    # A 마커로 분리
    q_pattern = re.compile(r"\nQ\s*:\s*", re.IGNORECASE)
    a_pattern = re.compile(r"\nA\s*:\s*", re.IGNORECASE)
    
    q_match = q_pattern.search(content)
    a_match = a_pattern.search(content)
    
    if q_match and a_match:
        question = content[q_match.end():a_match.start()].strip()
        answer = content[a_match.end():].strip()
    elif q_match and not a_match:
        # Q만 있고 A가 없는 경우
        question = content[q_match.end():].strip()
    elif not q_match and a_match:
        # Q 마커 없이 A만 있는 경우 - 전체를 Q로 사용
        question = content[:a_match.start()].strip()
        answer = content[a_match.end():].strip()
    else:
        # Q/A 마커 모두 없음 - 전체를 질문으로
        question = content.strip()
    
    # 제목이 Q에 포함되어 있으면 제거
    if title and question.startswith(title):
        question = question[len(title):].strip()
    
    return title, question, answer


# --- 전체 71852 데이터 파싱 ---
records_71852 = []
parse_failures = 0
no_answer = 0

all_71852_files = [(f, "label") for f in label_files] + [(f, "source") for f in source_files]

for filepath, file_type in tqdm(all_71852_files, desc="71852 파싱"):
    rec = load_71852_file(filepath)
    if rec is None:
        parse_failures += 1
        continue
    
    content = rec.get("consulting_content", "")
    raw_category = rec.get("consulting_category", "")
    source_region = rec.get("source", "")
    source_id = rec.get("source_id", "")
    filename = os.path.basename(filepath).replace(".json", "")
    
    title, question, answer = parse_consulting_content(content)
    
    if not answer or len(answer) < 30:
        no_answer += 1
        continue
    
    if not question or len(question) < 10:
        continue
    
    category = map_category(raw_category)
    
    record = {
        "id": f"71852_{file_type}_{filename}",
        "question": question,
        "answer": answer,
        "title": title,
        "category": category,
        "raw_category": raw_category,
        "source_region": source_region,
        "source_id": source_id,
        "source_dataset": f"71852_{file_type}",
        "q_len": len(question),
        "a_len": len(answer),
    }
    records_71852.append(record)

df_71852 = pd.DataFrame(records_71852)

print(f"\n=== 71852 파싱 결과 ===")
print(f"총 파일: {len(all_71852_files)}")
print(f"파싱 성공: {len(records_71852)}")
print(f"파싱 실패: {parse_failures}")
print(f"답변 부재/초단문(<30자): {no_answer}")
print(f"\n답변 길이 통계:")
print(df_71852["a_len"].describe())
print(f"\n질문 길이 통계:")
print(df_71852["q_len"].describe())
print(f"\n카테고리 분포:")
print(df_71852["category"].value_counts())

wandb.log({
    "71852_parsed_records": len(records_71852),
    "71852_avg_answer_len": df_71852["a_len"].mean(),
    "71852_avg_question_len": df_71852["q_len"].mean(),
})

## 3. 71852 데이터 처리 (Primary)

핵심 처리: `consulting_content`에서 Q와 A를 파싱하여 **전문 정부 답변**(200-500자)을 참조 답변으로 사용합니다.

기존 파이프라인이 사용한 `instructions.data.output`(평균 3-10자 추출형 답변)이 아닌, `consulting_content`의 A 섹션 전체를 사용합니다.

In [ ]:
# ============================================================
# Cell 4: Data Exploration - 98 (다산콜센터) & 71844 (금융 콜센터)
# ============================================================

# --- Dataset 98: 다산콜센터 ---
print("=== Dataset 98 (다산콜센터) ===")
dataset_98_files = glob.glob(os.path.join(DATASET_98_LABEL, "*.json"))
print(f"파일 수: {len(dataset_98_files)}")

dataset_98_stats = {}
for fpath in dataset_98_files:
    fname = os.path.basename(fpath)
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        # 카테고리 추출
        if isinstance(data, list) and len(data) > 0:
            cat = data[0].get("카테고리", "unknown")
            dataset_98_stats[fname] = {"count": len(data), "category": cat}
            print(f"  {fname}: {len(data):,}건, 카테고리: {cat}")
            # 샘플 출력
            sample_rec = data[0]
            print(f"    키: {list(sample_rec.keys())[:8]}")
            q = sample_rec.get("고객질문(요청)", "")
            if q:
                print(f"    Q 예시: {q[:100]}")
    except Exception as e:
        print(f"  {fname}: 로드 실패 - {e}")

# --- Dataset 71844: 금융 콜센터 (제거 대상) ---
print(f"\n=== Dataset 71844 (금융 콜센터) - 제거 대상 ===")
dataset_71844_files = glob.glob(os.path.join(DATASET_71844_LABEL, "*.json"))
print(f"파일 수: {len(dataset_71844_files)}")
for fpath in dataset_71844_files[:2]:
    fname = os.path.basename(fpath)
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f"  {fname}: {len(data):,}건")
            if data:
                print(f"    키: {list(data[0].keys())[:8]}")
    except Exception as e:
        print(f"  {fname}: 로드 실패 - {e}")

print("\n** 71844 (LGU+/하나카드) 데이터는 금융 상담이므로 전량 제거 **")

In [ ]:
# ============================================================
# Cell 3: Data Exploration - 619 (Supplementary, Q-only)
# ============================================================

print("=== Dataset 619 (민원 업무 자동화 AI 언어 데이터) ===")
print("Q-only 데이터셋 - 분류 학습 및 입력 예시용\n")

dataset_619_stats = {}
if os.path.exists(DATASET_619_LABEL):
    categories_619 = sorted(os.listdir(DATASET_619_LABEL))
    print(f"카테고리 수: {len(categories_619)}")
    
    total_docs = 0
    for cat_dir in tqdm(categories_619, desc="619 카테고리 스캔"):
        cat_path = os.path.join(DATASET_619_LABEL, cat_dir)
        if not os.path.isdir(cat_path):
            continue
        cat_files = glob.glob(os.path.join(cat_path, "*.json"))
        doc_count = 0
        sample_q = ""
        for cf in cat_files:
            try:
                with open(cf, "r", encoding="utf-8") as f:
                    cdata = json.load(f)
                docs = cdata.get("documents", [])
                doc_count += len(docs)
                if docs and not sample_q:
                    sample_q = docs[0].get("Q_refined", "")[:100]
            except:
                pass
        dataset_619_stats[cat_dir] = {"count": doc_count, "sample": sample_q}
        total_docs += doc_count
    
    print(f"\n총 문서 수: {total_docs:,}")
    print(f"\n카테고리별 분포:")
    for cat, info in sorted(dataset_619_stats.items(), key=lambda x: -x[1]["count"]):
        print(f"  {cat}: {info['count']:,} ({info['count']/total_docs*100:.1f}%)")
        if info["sample"]:
            print(f"    예시: {info['sample']}")
    
    wandb.log({"619_total_docs": total_docs, "619_categories": len(categories_619)})
else:
    print("619 데이터셋 경로 없음")

In [ ]:
# ============================================================
# Cell 2: Data Exploration - 71852 (Primary Dataset)
# ============================================================

def load_71852_file(filepath: str) -> Optional[dict]:
    """71852 JSON 파일 1개를 로드하여 첫 번째 레코드 반환."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list) and len(data) > 0:
            return data[0]
        return data
    except Exception as e:
        return None

# --- Label 샘플 확인 ---
label_files = sorted(glob.glob(os.path.join(DATASET_71852_LABEL, "*.json")))
source_files = sorted(glob.glob(os.path.join(DATASET_71852_SOURCE, "*.json")))

print(f"=== Dataset 71852 ===")
print(f"Label files: {len(label_files)}")
print(f"Source files: {len(source_files)}")
print(f"Total files: {len(label_files) + len(source_files)}")

# 샘플 1개 구조 확인
sample = load_71852_file(label_files[0])
if sample:
    print(f"\n--- Label 샘플 키 ---")
    print(f"Keys: {list(sample.keys())}")
    print(f"source: {sample.get('source')}")
    print(f"consulting_category: {sample.get('consulting_category')}")
    print(f"consulting_content 길이: {len(sample.get('consulting_content', ''))}")
    print(f"\n--- consulting_content 앞 500자 ---")
    print(sample.get("consulting_content", "")[:500])
    
    # instructions 구조
    instructions = sample.get("instructions", [])
    if instructions:
        print(f"\n--- instructions 구조 ---")
        print(f"tuning_type: {instructions[0].get('tuning_type')}")
        inst_data = instructions[0].get("data", [{}])[0]
        print(f"instruction output (ultra-short): '{inst_data.get('output', '')}'")
        print(f"output 길이: {len(inst_data.get('output', ''))}")

# Source 샘플 확인
sample_src = load_71852_file(source_files[0])
if sample_src:
    print(f"\n--- Source 샘플 ---")
    print(f"source: {sample_src.get('source')}")
    print(f"consulting_category: {sample_src.get('consulting_category')}")

# 카테고리 분포 확인
categories_label = []
categories_source = []

print("\nLabel 카테고리 스캔 중...")
for f in tqdm(label_files, desc="71852 Label"):
    rec = load_71852_file(f)
    if rec:
        categories_label.append(rec.get("consulting_category", "unknown"))

print("\nSource 카테고리 스캔 중...")
for f in tqdm(source_files, desc="71852 Source"):
    rec = load_71852_file(f)
    if rec:
        categories_source.append(rec.get("consulting_category", "unknown"))

all_categories = categories_label + categories_source
cat_counter = Counter(all_categories)

print(f"\n=== 71852 카테고리 분포 (총 {len(all_categories)}건) ===")
for cat, count in cat_counter.most_common():
    print(f"  {cat}: {count} ({count/len(all_categories)*100:.1f}%)")

wandb.log({"71852_total_files": len(label_files) + len(source_files),
           "71852_unique_categories": len(cat_counter)})

## 2. 데이터 탐색 (Data Exploration)

각 데이터셋의 원본 구조와 통계를 확인합니다.

In [ ]:
# ============================================================
# Cell 1: Setup & Configuration
# ============================================================
!pip install -q wandb tqdm pandas matplotlib seaborn

import os
import json
import glob
import re
import hashlib
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# ---- Google Drive Mount (Colab) ----
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
    BASE_DIR = "/content/drive/MyDrive/ondevice-ai-civil-complaint"
except ImportError:
    IS_COLAB = False
    BASE_DIR = ".."  # 로컬 실행 시

# ---- Paths ----
RAW_DIR = os.path.join(BASE_DIR, "data/raw/aihub")
DATASET_71852_LABEL = os.path.join(RAW_DIR, "71852/label")
DATASET_71852_SOURCE = os.path.join(RAW_DIR, "71852/source")
DATASET_619_LABEL = os.path.join(RAW_DIR, "619/label")
DATASET_98_LABEL = os.path.join(RAW_DIR, "98/label")
DATASET_71844_LABEL = os.path.join(RAW_DIR, "71844/label")

OUTPUT_DIR = os.path.join(BASE_DIR, "data/processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- W&B Init ----
import wandb
wandb.login()
run = wandb.init(
    project="govon-data-reconstruction",
    name="data-reconstruction-v2",
    tags=["data-quality", "reconstruction"],
    config={
        "primary_dataset": "71852",
        "supplementary_dataset": "619",
        "removed_datasets": ["71844"],
        "filtered_datasets": ["98"],
    }
)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"71852 label files: {len(glob.glob(os.path.join(DATASET_71852_LABEL, '*.json')))}") 
print(f"71852 source files: {len(glob.glob(os.path.join(DATASET_71852_SOURCE, '*.json')))}")
print(f"619 categories: {os.listdir(DATASET_619_LABEL) if os.path.exists(DATASET_619_LABEL) else 'N/A'}")
print(f"98 label files: {len(glob.glob(os.path.join(DATASET_98_LABEL, '*.json')))}")
print("Setup complete.")

# 01. 민원 데이터 재구성 (Data Reconstruction)

## 목적
BLEU/ROUGE-L 성능 개선을 위한 학습 데이터 품질 전면 재구축

## 현재 데이터 문제점
1. **카테고리 다양성 제로**: 15,604건 전체가 "other"로 분류
2. **금융 콜센터 데이터 오염**: 73.5% (9,169건)가 LGU+/하나카드 등 금융 상담 (datasets 71844/98)
3. **참조 답변 초단문화**: 평균 97자 - BLEU/ROUGE 왜곡
4. **[NAME_MASKED] 과다**: 밀도 23.35% - 평가 메트릭 오염
5. **`<thought>` 태그 동일**: 전 레코드 동일한 thought - 학습 신호 제로
6. **71852 데이터셋 미활용**: 13,000+ 고품질 민원 Q&A 쌍 존재

## 재구성 전략
- **71852** (label 9,918 + source 3,280): 실제 민원 Q&A 전수 활용, `consulting_content`에서 전문 답변 추출
- **619** (18개 카테고리, 80만건): Q-only, 분류 학습 및 입력 예시용 샘플링
- **71844** (금융 콜센터): **완전 제거**
- **98** (다산콜센터): 민원 관련 건만 필터링